## Limpeza e preparação dos dados
## 1. carregar o dataset e mostrar as informações

In [8]:
import pandas as pd
import numpy as np
df = pd.read_csv('Preços semestrais - AUTOMOTIVOS_2025.02.csv', sep=';', encoding='utf-8')
print("--- Informações do Dataset ---")
display(df.info())
print("\n--- Valores Nulos por Coluna ---")
display(df.isnull().sum())

--- Informações do Dataset ---
<class 'pandas.DataFrame'>
RangeIndex: 384208 entries, 0 to 384207
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Regiao - Sigla     384208 non-null  str    
 1   Estado - Sigla     384208 non-null  str    
 2   Municipio          384208 non-null  str    
 3   Revenda            384208 non-null  str    
 4   CNPJ da Revenda    384208 non-null  int64  
 5   Nome da Rua        384208 non-null  str    
 6   Numero Rua         384184 non-null  str    
 7   Complemento        86707 non-null   str    
 8   Bairro             383546 non-null  str    
 9   Cep                384208 non-null  str    
 10  Produto            384208 non-null  str    
 11  Data da Coleta     384208 non-null  str    
 12  Valor de Venda     384208 non-null  str    
 13  Valor de Compra    0 non-null       float64
 14  Unidade de Medida  384208 non-null  str    
 15  Bandeira           384208 non-n

None


--- Valores Nulos por Coluna ---


Regiao - Sigla            0
Estado - Sigla            0
Municipio                 0
Revenda                   0
CNPJ da Revenda           0
Nome da Rua               0
Numero Rua               24
Complemento          297501
Bairro                  662
Cep                       0
Produto                   0
Data da Coleta            0
Valor de Venda            0
Valor de Compra      384208
Unidade de Medida         0
Bandeira                  0
dtype: int64

## 2.Tratamento dos Dados e preparação para análise

In [9]:
#copia para nao alterar os dados originais
df_clean = df.copy()
#remover colunas desnecessarias para as perguntas
colunas_remover = ['Valor de Compra', 'Complemento', 'Nome da Rua', 'Numero Rua', 'Cep', 'CNPJ da Revenda']
df_clean = df_clean.drop(columns=colunas_remover)
#ajustar o valor de venda e transformar em float
df_clean['Valor de Venda'] = df_clean['Valor de Venda'].astype(str).str.replace(',', '.').astype(float)
#transformar data no formato datetime
df_clean['Data da Coleta'] = pd.to_datetime(df_clean['Data da Coleta'], format='%d/%m/%Y')
#criar a coluna mes
df_clean['Mes'] = df_clean['Data da Coleta'].dt.month
#lidar com dados faltantes
df_clean = df_clean.dropna(subset=['Valor de Venda', 'Produto'])
#verificar o resultado pos limpeza
print("--- Dimensões do dataset limpo ---")
print(df_clean.shape)
display(df_clean.head())

--- Dimensões do dataset limpo ---
(384208, 11)


,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,Bairro,Produto,Data da Coleta,Valor de Venda,Unidade de Medida,Bandeira,Mes
0,NE,AL,DELMIRO GOUVEIA,COMERCIO DE COMBUSTIVEIS NENZITA LTDA,PEDRA VELHA,DIESEL,2025-07-01,6.69,R$ / litro,BRANCA,7
1,N,AP,MACAPA,SEPE TIARAJU EMPREENDIMENTOS LTDA,ARAXÁ,DIESEL,2025-07-01,6.33,R$ / litro,BRANCA,7
2,NE,BA,IPIRA,AUTO POSTO AUGUSTU'S LTDA,SEDE,DIESEL,2025-07-01,6.90,R$ / litro,ALE,7
3,NE,BA,ITAMARAJU,CODEPEL -COMERCIO DE DERIVADOS DE PETROLEO LTDA,CENTRO,DIESEL,2025-07-01,5.99,R$ / litro,BRANCA,7
4,NE,BA,ITAMARAJU,BENTIVI DERIVADOS DE PETROLEO LTDA,SEDE,DIESEL,2025-07-01,5.92,R$ / litro,IPIRANGA,7


## 2.1Tratamento dos dados do dataset da frota de veiculos

In [10]:
#segundo dataset
#carregar o segundo dataset
df_frota = pd.read_csv('frota_dezembro_2025.csv', sep=';', encoding='utf-8')
# display inicial para ve como as colunas estão configuradas
display(df_frota.head())
#print para ver o nome da exato das colunas
print(df_frota.columns.tolist())

,UF,MunicÃ­pio,Marca Modelo,Ano FabricaÃ§Ã£o VeÃ­culo CRV,Qtd. VeÃ­culos
0,ACRE,ACRELANDIA,AGRALE/13000,2009,1.0
1,ACRE,ACRELANDIA,AGRALE/1800,1989,1.0
2,ACRE,ACRELANDIA,AGRALE/1800,1990,1.0
3,ACRE,ACRELANDIA,AGRALE/1800D RD,1989,1.0
4,ACRE,ACRELANDIA,AGRALE/1800D RD,1990,2.0


['UF', 'MunicÃ\xadpio', 'Marca Modelo', 'Ano FabricaÃ§Ã£o VeÃ\xadculo CRV', 'Qtd. VeÃ\xadculos']


In [11]:
#limpando a coluna municipio
df_clean['Municipio'] = df_clean['Municipio'].str.upper().str.strip()
df_clean['Municipio'] = df_clean['Municipio'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
#limpando a coluna com nome quebrado
df_frota['MunicÃ\xadpio'] = df_frota['MunicÃ\xadpio'].str.upper().str.strip()
df_frota['MunicÃ\xadpio'] = df_frota['MunicÃ\xadpio'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')

In [6]:
#Mudando o nome quebrado das colunas
df_frota = df_frota.rename(columns={
    'UF': 'Estado - Sigla',
    'MunicÃ\xadpio': 'Municipio',
    'Qtd. VeÃ\xadculos': 'Qtd_Veiculos'
})
# Limpeza no dataset de combustiveis
df_clean['Municipio'] = df_clean['Municipio'].astype(str).str.upper().str.strip()
df_clean['Municipio'] = df_clean['Municipio'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_clean['Estado - Sigla'] = df_clean['Estado - Sigla'].astype(str).str.upper().str.strip()
# Limpeza no dataset da frota nacional
df_frota['Municipio'] = df_frota['Municipio'].astype(str).str.upper().str.strip()
df_frota['Municipio'] = df_frota['Municipio'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_frota['Estado - Sigla'] = df_frota['Estado - Sigla'].astype(str).str.upper().str.strip()
#Como os datasets tratam o nome dos estados de forma diferente, vamos ter que modificar TODOS os nomes dos estados do dataset
dicionario_estados = {
    'ACRE': 'AC', 'ALAGOAS': 'AL', 'AMAPA': 'AP', 'AMAZONAS': 'AM', 'BAHIA': 'BA',
    'CEARA': 'CE', 'DISTRITO FEDERAL': 'DF', 'ESPIRITO SANTO': 'ES', 'GOIAS': 'GO',
    'MARANHAO': 'MA', 'MATO GROSSO': 'MT', 'MATO GROSSO DO SUL': 'MS', 'MINAS GERAIS': 'MG',
    'PARA': 'PA', 'PARAIBA': 'PB', 'PARANA': 'PR', 'PERNAMBUCO': 'PE', 'PIAUI': 'PI',
    'RIO DE JANEIRO': 'RJ', 'RIO GRANDE DO NORTE': 'RN', 'RIO GRANDE DO SUL': 'RS',
    'RONDONIA': 'RO', 'RORAIMA': 'RR', 'SANTA CATARINA': 'SC', 'SAO PAULO': 'SP',
    'SERGIPE': 'SE', 'TOCANTINS': 'TO'
}
#usando o dicionario_estados, concluimos a troca para padronizar a tabela
df_frota['Estado - Sigla'] = df_frota['Estado - Sigla'].replace(dicionario_estados)
# Agrupar o dataset e merge
frota_municipal = df_frota.groupby(['Estado - Sigla', 'Municipio'])['Qtd_Veiculos'].sum().reset_index()
frota_municipal = frota_municipal.rename(columns={'Qtd_Veiculos': 'Frota_da_Cidade'})
df_final_detalhado = pd.merge(df_clean, frota_municipal, on=['Estado - Sigla', 'Municipio'], how='left')
display(df_final_detalhado.head())

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,Bairro,Produto,Data da Coleta,Valor de Venda,Unidade de Medida,Bandeira,Mes,Frota_da_Cidade
0,NE,AL,DELMIRO GOUVEIA,COMERCIO DE COMBUSTIVEIS NENZITA LTDA,PEDRA VELHA,DIESEL,2025-07-01,6.69,R$ / litro,BRANCA,7,30082.0
1,N,AP,MACAPA,SEPE TIARAJU EMPREENDIMENTOS LTDA,ARAXÁ,DIESEL,2025-07-01,6.33,R$ / litro,BRANCA,7,215772.0
2,NE,BA,IPIRA,AUTO POSTO AUGUSTU'S LTDA,SEDE,DIESEL,2025-07-01,6.90,R$ / litro,ALE,7,25530.0
3,NE,BA,ITAMARAJU,CODEPEL -COMERCIO DE DERIVADOS DE PETROLEO LTDA,CENTRO,DIESEL,2025-07-01,5.99,R$ / litro,BRANCA,7,30777.0
4,NE,BA,ITAMARAJU,BENTIVI DERIVADOS DE PETROLEO LTDA,SEDE,DIESEL,2025-07-01,5.92,R$ / litro,IPIRANGA,7,30777.0


In [7]:
#como utilizei dados da frota nacional de dezembro, vou colocar apenas o mes 12 para ser analisado
#copia do dataset final, porem, apenas com o mes 12
df_dezembro = df_final_detalhado[df_final_detalhado['Mes'] == 12].copy()
#comparar o numero de linhas de cada um.
print("--- Quantidade de Linhas ---")
print("Dataset Original (Julho a Dezembro):", len(df_final_detalhado))
print("Dataset Filtrado (Apenas Dezembro):", len(df_dezembro))
#print final com os dados apenas do ultimo mes.
print("\n--- Amostra dos Dados de Dezembro ---")
display(df_dezembro.head())

--- Quantidade de Linhas ---
Dataset Original (Julho a Dezembro): 384208
Dataset Filtrado (Apenas Dezembro): 75580

--- Amostra dos Dados de Dezembro ---


,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,Bairro,Produto,Data da Coleta,Valor de Venda,Unidade de Medida,Bandeira,Mes,Frota_da_Cidade
32056,N,AP,MACAPA,MACHADO & ANDRADE LTDA,JARDIM MARCO ZERO,DIESEL,2025-12-01,6.39,R$ / litro,IPIRANGA,12,215772.0
32057,NE,BA,JEQUIE,D & M COMERCIAL LTDA,CENTRO,DIESEL,2025-12-01,6.50,R$ / litro,RAIZEN,12,86288.0
32058,NE,BA,JEQUIE,JEQUIEZINHO COMERCIO VAREJISTA DE COMBUSTIVEIS...,JEQUIEZINHO,DIESEL,2025-12-01,6.05,R$ / litro,BRANCA,12,86288.0
32059,NE,BA,JEQUIE,COM DE DER DE PETROLEO BARRETO LTDA,CENTRO,DIESEL,2025-12-01,6.05,R$ / litro,IPIRANGA,12,86288.0
32060,NE,BA,JEQUIE,M W COMERCIO DE COMBUSTIVEIS SERVICOS E ALIMEN...,JEQUIEZINHO,DIESEL,2025-12-01,6.29,R$ / litro,BRANCA,12,86288.0
